# Analiza e të Dhënave të Krimit – Los Angeles (2020 deri tani)

## Përshkrim i Projektit
Ky notebook analizon dataset-in e krimeve të raportuara nga Departamenti i Policisë së Los Angeles (LAPD) nga viti 2020 deri në ditët e sotme.  
Dataset-i përmban mbi **1 milion rekorde** dhe **28 kolona**, duke mbuluar informacione si: lloji i krimit, zona, koha, viktimat, armët e përdorura dhe statusi i çështjes.

**Burimi:** [Los Angeles Open Data Portal](https://data.lacity.org/)

---

## 1. Importimi i Librarive

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print('Librarite u importuan me sukses.')

Librarite u importuan me sukses.


## 2. Ngarkimi i Dataset-it

In [5]:
# Ngarkojmë dataset-in
FILE_PATH = 'Crime_Data_from_2020_to_Present.csv'

df = pd.read_csv(FILE_PATH, low_memory=False)

print(f'Dimensionet e dataset-it: {df.shape}')
print(f'Numri i kolonave: {df.shape[1]}')
print(f'Numri i rekordeve: {df.shape[0]:,}')
df.head(3)

Dimensionet e dataset-it: (1005198, 28)
Numri i kolonave: 28
Numri i rekordeve: 1,005,198


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,190326475,3/1/2020 0:00,3/1/2020 0:00,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,NaN,0,M,O,101.0,STREET,NaN,NaN,AA,Adult Arrest,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506
1,200106753,2/9/2020 0:00,2/8/2020 0:00,1800,1,Central,182,1,330,BURGLARY FROM VEHICLE,1822 1402 0344,47,M,O,128.0,BUS STOP/LAYOVER (ALSO QUERY 124),NaN,NaN,IC,Invest Cont,330.0,998.0,NaN,NaN,1000 S FLOWER ST,NaN,34.0444,-118.2628
2,200320258,11/11/2020 0:00,11/4/2020 0:00,1700,3,Southwest,356,1,480,BIKE - STOLEN,0344 1251,19,X,X,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",NaN,NaN,IC,Invest Cont,480.0,NaN,NaN,NaN,1400 W 37TH ST,NaN,34.0210,-118.3002


## 3. Të dhënat kryesore mbi datasetin

In [6]:
# Tipet e të dhënave
print('=== Tipet e kolonave ===')
print(df.dtypes)
print()

# Statistika përshkruese për kolonat numerike
print('=== Statistika Përshkruese ===')
df[['Vict Age', 'TIME OCC']].describe()

=== Tipet e kolonave ===
DR_NO               int64
Date Rptd             str
DATE OCC              str
TIME OCC            int64
AREA                int64
AREA NAME             str
Rpt Dist No         int64
Part 1-2            int64
Crm Cd              int64
Crm Cd Desc           str
Mocodes               str
Vict Age            int64
Vict Sex              str
Vict Descent          str
Premis Cd         float64
Premis Desc           str
Weapon Used Cd    float64
Weapon Desc           str
Status                str
Status Desc           str
Crm Cd 1          float64
Crm Cd 2          float64
Crm Cd 3          float64
Crm Cd 4          float64
LOCATION              str
Cross Street          str
LAT               float64
LON               float64
dtype: object

=== Statistika Përshkruese ===


,Vict Age,TIME OCC
count,1.005198e+06,1.005198e+06
mean,2.891253e+01,1.339911e+03
std,2.199382e+01,6.510531e+02
min,-4.000000e+00,1.000000e+00
25%,0.000000e+00,9.000000e+02
50%,3.000000e+01,1.420000e+03
75%,4.400000e+01,1.900000e+03
max,1.200000e+02,2.359000e+03


In [ ]:
# Vlerat munguese për çdo kolonë
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Vlera Mungese': missing,
    'Përqindja (%)': missing_pct
}).query('`Vlera Mungese` > 0').sort_values('Përqindja (%)', ascending=False)

print('=== Kolonat me vlera mungese ===')
print(missing_df)


for col in df.columns:
    print(f"\nColumn {col} has {df[col].isnull().sum()} null values")

=== Kolonat me vlera mungese ===
                Vlera Mungese  Përqindja (%)
Crm Cd 4              1005134          99.99
Crm Cd 3              1002884          99.77
Crm Cd 2               936039          93.12
Cross Street           850955          84.66
Weapon Desc            677918          67.44
Weapon Used Cd         677918          67.44
Mocodes                151760          15.10
Vict Descent           144794          14.40
Vict Sex               144782          14.40
Premis Desc               588           0.06
Premis Cd                  16           0.00
Crm Cd 1                   11           0.00
Status                      1           0.00

Column DR_NO has 0 null values

Column Date Rptd has 0 null values

Column DATE OCC has 0 null values

Column TIME OCC has 0 null values

Column AREA has 0 null values

Column AREA NAME has 0 null values

Column Rpt Dist No has 0 null values

Column Part 1-2 has 0 null values

Column Crm Cd has 0 null values

Column Crm Cd Desc has 0 

## Sqarim për Emrat e Kolonave

| Kolona | Përshkrim |
|--------|-----------|
| DR_NO | Unique identifier for the crime report |
| Date Rptd | The date the crime was reported |
| DATE OCC | The actual date the crime occurred |
| TIME OCC | The time the crime occurred, usually in 24-hour format |
| AREA | Numeric code representing the geographical area where the crime occurred |
| AREA NAME | Name of the geographical area |
| Rpt Dist No | Reporting district number for the incident |
| Part 1-2 | Crime classification (e.g., Part 1 for serious crimes, Part 2 for less serious crimes) |
| Crm Cd | Numeric code representing the type of crime |
| Crm Cd Desc | Description of the crime type |
| Mocodes | Modus operandi codes, describing the method used in the crime |
| Vict Age | Age of the victim |
| Vict Sex | Gender of the victim (e.g., Male, Female, Unknown) |
| Vict Descent | Ethnicity or descent of the victim |
| Premis Cd | Numeric code for the type of premises where the crime occurred |
| Premis Desc | Description of the type of premises (e.g., residence, vehicle, commercial) |
| Weapon Used Cd | Numeric code for the weapon used in the crime, if applicable |
| Weapon Desc | Description of the weapon used |
| Status | Status code of the crime case (e.g., Open, Solved) |
| Status Desc | Description of the case status |
| Crm Cd 1 | Additional crime codes, if the incident involved multiple offenses |
| Crm Cd 2 | Additional crime codes, if the incident involved multiple offenses |
| Crm Cd 3 | Additional crime codes, if the incident involved multiple offenses |
| Crm Cd 4 | Additional crime codes, if the incident involved multiple offenses |
| LOCATION | Text description of the crime location |
| Cross Street | Nearby cross street for the crime location |
| LAT | Latitude of the crime location |
| LON | Longitude of the crime location |

## 4. Pastrimi i të Dhënave dhe Vendimmarrja për Vlerat Munguese

### Strategjia:
| Kolona | Vlera Mungese | Vendimi | Arsyeja |
|---|---|---|---|
| `Weapon Used Cd` / `Weapon Desc` | ~67% | Zëvendëso me `'UNKNOWN'` / `0` | Mungesa tregon krim pa armë |
| `Vict Sex` / `Vict Descent` | ~14% | Zëvendëso me `'X'` (E panjohur) | Kodi zyrtar i LAPD-së |
| `Mocodes` | ~15% | Zëvendëso me `'NONE'` | Kode shtesë opsionale |
| `Crm Cd 2/3/4` | >93% | Mbaj si NaN / shpërnore | Kolona sekondare, jo kyçe |
| `Cross Street` | ~85% | Mbaj si NaN | Jo e nevojshme për analiza |
| `Premis Desc` / `Premis Cd` / `Crm Cd 1` / `Status` | <0.1% | Zëvendëso me `'UNKNOWN'` | Shumë pak mungesa |

In [ ]:
# --- Konvertimi i datave ---
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], format='%m/%d/%Y %H:%M', errors='coerce')
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %H:%M', errors='coerce')

# Nxjerrja e veçorive kohore
df['Year']  = df['DATE OCC'].dt.year
df['Month'] = df['DATE OCC'].dt.month
df['Hour']  = (df['TIME OCC'] // 100)  # TIME OCC format: HHMM
df['DayOfWeek'] = df['DATE OCC'].dt.day_name()

# --- Pastrimi i kolonave kategorike ---
df['Weapon Desc']   = df['Weapon Desc'].fillna('UNKNOWN')
df['Weapon Used Cd']= df['Weapon Used Cd'].fillna(0)
df['Vict Sex']      = df['Vict Sex'].fillna('X').replace({'': 'X', '-': 'X'})
df['Vict Descent']  = df['Vict Descent'].fillna('X').replace({'': 'X', '-': 'X'})
df['Mocodes']       = df['Mocodes'].fillna('NONE')
df['Premis Desc']   = df['Premis Desc'].fillna('UNKNOWN')

# --- Pastrimi i moshës (vlera negative ose jashtë intervalit logjik) ---
df['Vict Age'] = df['Vict Age'].replace(0, np.nan)  # 0 = e panjohur
df.loc[df['Vict Age'] < 0, 'Vict Age'] = np.nan
df.loc[df['Vict Age'] > 99, 'Vict Age'] = np.nan

# --- Kolona ndihmëse: ka armë apo jo ---
df['Has Weapon'] = (df['Weapon Desc'] != 'UNKNOWN').astype(int)

# --- Kolona: rast i zgjidhur apo jo ---
df['Solved'] = df['Status Desc'].isin(['Adult Arrest', 'Juv Arrest']).astype(int)

print(f'Dataset pas pastrimit: {df.shape}')
print(f'Vlera NaN në Vict Age pas pastrimit: {df["Vict Age"].isnull().sum():,}')
print('Pastrimi u krye me sukses!')

Dataset pas pastrimit: (1005198, 34)
Vlera NaN në Vict Age pas pastrimit: 269,514
Pastrimi u krye me sukses!


---
## 7. Referencat

1. Los Angeles Police Department – Open Data Portal: https://data.lacity.org/Public-Safety/Crime-Data-from-2020-to-Present/2nrs-mtv8